## RAG 데이터 수집 코드 작성하기

앞서 실습한 RAG 프로세스를 프로젝트 에이전트 코드에 추가해 봅시다.

### 1. 다양한 형식의 파일들 입력 받아서 Document로 변환하고 split하기

앞서 살펴 본 다양한 형식의 파일들을 텍스트로 변환한 뒤, 적절한 크기로 split 하는 함수를 작성해 봅시다!

어떤 유형의 문서가 들어올 지 알 수 없으므로, 함수 하나를 이용해 다양한 형식의 파일들을 한번에 처리하는 함수로 만들어 보겠습니다.

In [1]:
# excel 파일을 입력 받아 텍스트로 변환하고 적절한 크기로 split 하는 함수

import pandas as pd
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def parse_excel(file_path: str) -> list[Document]:
    # 1. 전달받은 경로(file_path)에 있는 엑셀 파일을 읽어옵니다.
    with pd.ExcelFile(file_path) as xl:
    # 2. 데이터를 담아둘 빈 바구니(리스트)를 만듭니다.
        docs = []
        
        # 3. 엑셀 파일 안에 있는 여러 개의 '시트(sheet)'를 하나씩 꺼내서 반복합니다.
        for sheet_name in xl.sheet_names:
            # 4. 현재 시트의 내용을 표(데이터프레임) 형태로 불러옵니다.
            # header=1 은 엑셀의 두 번째 줄(행)을 제목(컬럼 이름)으로 사용한다는 뜻입니다.
            df = xl.parse(sheet_name, header=1)
            
            # 5. 표의 데이터를 한 줄씩 꺼내서 반복합니다.
            for idx, row in df.iterrows():
                # 6. 한 줄에 있는 각 칸의 데이터를 "제목: 내용" 형식으로 묶어서 하나의 긴 글로 만듭니다.
                # (예: "이름: 홍길동 \n 나이: 20")
                text = "\n".join(f"{col}: {row[col]}" for col in df.columns)
                
                # 7. 만들어진 글과 출처 정보를 하나로 묶어서(Document 객체) 바구니에 담습니다.
                docs.append(Document(
                    page_content=text, # 실제 내용이 들어가는 부분
                    metadata={         # 이 데이터가 어디서 왔는지 기록하는 부분
                        "source": file_path, # 어떤 파일에서 왔는지
                        "sheet": sheet_name, # 어떤 시트에서 왔는지
                        "row": idx,          # 몇 번째 줄(행)에서 왔는지
                    }
                ))
                
    # 8. 모든 데이터가 담긴 바구니(문서 리스트)를 최종적으로 반환합니다.
    return docs

In [2]:
# PDF를 입력 받아 텍스트로 변환하고 적절한 크기로 split 하는 함수

from langchain_community.document_loaders import PyPDFLoader


def parse_pdf(file_path: str) -> list[Document]:
    # pdf 파일을 불러오는 코드
    pdf_contents = PyPDFLoader(file_path).load()

    return pdf_contents

In [3]:
# word를 입력 받아 텍스트로 변환하는 함수

from langchain_community.document_loaders import Docx2txtLoader


def parse_word(file_path: str) -> list[Document]:
    # word 파일을 불러오는 코드
    word_contents = Docx2txtLoader(file_path).load()

    return word_contents

In [4]:
# ppt를 입력 받아 텍스트로 변환하는 함수

from pptx import Presentation


def parse_pptx(file_path: str) -> list[Document]:
    # ppt 파일 로드
    prs = Presentation(file_path)
    docs = []
    # 각 슬라이드를 돌며...
    for slide_num, slide in enumerate(prs.slides, 1):
        texts = []
        # 슬라이드의 모든 요소를 살펴보며 텍스트가 있으면 texts 리스트에 추가.
        for shape in slide.shapes:
            if shape.has_text_frame:
                for para in shape.text_frame.paragraphs:
                    if para.text.strip():
                        texts.append(para.text.strip())
        if texts:
            docs.append(Document(
                page_content="\n".join(texts),
                metadata={
                    "source": file_path,
                    "slide": slide_num,
                }
            ))
    return docs

다음으로는 텍스트 문서들을 적절한 크기의 chunk로 분할하는 함수도 구현해 보겠습니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(docs: list[Document], chunk_size: int, chunk_overlap: int) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(docs)
    return chunks

이제 위에서 구현한 함수들을 이용해서 아래와 같은 기능을 갖는 함수를 구현해 봅시다!

1. new_docs 폴더에 있는 파일들을 모두 불러온다.
2. 모든 파일의 확장자를 보고 word, excel, pdf, ppt parser를 이용해 텍스트 문서로 변환한다.
3. 텍스트 문서로 변환된 파일을 split_documents를 이용해 적절한 크기로 나눈다.
4. FAISS 벡터로 변환한 뒤 'faiss_index' 폴더에 저장한다. (이미 저장된 벡터가 있다면 기존 벡터에 새로운 벡터를 추가해서 저장한다.)
5. 등록된 문서들은 registered_docs로 옮긴다.

In [ ]:
import os, shutil
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


load_dotenv(dotenv_path="../.env")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)


def register_rag_docs(new_docs: str = "new_docs", chunk_size: int = 500, chunk_overlap: int = 50):
    new_files = os.listdir(new_docs)
    print(f"총 {len(new_files)}개의 파일을 찾았습니다!")

    total_docs = []

    for new_file in new_files:
        file_path = os.path.join("new_docs", new_file)
        if file_path.endswith(".doc") or file_path.endswith(".docs") or file_path.endswith(".docx"):
            docs = parse_word(file_path)
        elif file_path.endswith(".xlsx"):
            docs = parse_excel(file_path)
        elif file_path.endswith(".pdf"):
            docs = parse_pdf(file_path)
        elif file_path.endswith(".ppt") or file_path.endswith(".pptx"):
            docs = parse_pptx(file_path)
        else:
            continue
        total_docs += docs

        registered_path = os.path.join("registered_docs", new_file)
        shutil.move(file_path, registered_path)
    
    chunks = split_documents(total_docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    print(f"총 {len(chunks)}개의 문서가 생성되었습니다!")
    vectorstore = FAISS.from_documents(chunks, OpenAIEmbeddings())

    # 이미 기존의 faiss 벡터가 존재한다면, 기존의 벡터에 추가
    if os.path.exists("faiss_index"):
        original_vectorstore = FAISS.load_local(
            "faiss_index",
            OpenAIEmbeddings(),
            allow_dangerous_deserialization=True
        )
        vectorstore.merge_from(original_vectorstore)

    vectorstore.save_local("faiss_index")

register_rag_docs()

총 4개의 파일을 찾았습니다!
총 124개의 문서가 생성되었습니다!


그럼 이제 구현한 코드들을 실제 프로젝트 코드에 옮겨 볼까요?

## 에이전트에 RAG 기능 추가하기

앞서 생성한 FAISS 임베딩 벡터를 사용해 앞서 구현했던 에이전트에게 RAG 기능을 더해 봅시다.

1. tools에 앞서 구현한 search_documents 함수를 추가합니다.
2. 생성된 faiss_index 폴더를 agent 폴더 안에 집어넣습니다.
3. search_documents에서 경로 문제가 발생한다면 함수 내부의 코드 일부를 아래와 같이 수정하여 경로 문제를 해결합니다.

```
    relative_path = os.path.relpath(os.path.join(current_dir, "faiss_index"))
    vectorstore = FAISS.load_local(
        relative_path,
        OpenAIEmbeddings(),
        allow_dangerous_deserialization=True
    )
```


4. agent.py에서 새롭게 추가된 tool을 추가합니다.

In [1]:
import os
from pathlib import Path
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
@tool
def search_documents(query: str) -> str:
    """VectorDB에서 문서를 검색합니다. 키키테크의 제품 정보, 사내 규정, 임직원 및 프로젝트 현황 등의 정보를 확인할 수 있습니다."""
    current_dir = os.path.dirname(__file__)
    # Windows 환경에서 FAISS가 한글 경로를 인식하지 못하는 버그를 우회하기 위해 상대 경로 사용
    relative_path = os.path.relpath(os.path.join(current_dir, "faiss_index"))
    vectorstore = FAISS.load_local(
        relative_path,
        OpenAIEmbeddings(),
        allow_dangerous_deserialization=True
    )
    results = vectorstore.similarity_search(query, k=5) # 상위 5개 문서 추출
    formatted = []
    for doc in results:
        meta = doc.metadata
        # Document 객체는 메타데이터를 포함하고 있어 참고한 문서의 출처를 확인할 수 있습니다.
        source = Path(meta.get("source", "unknown")).name
        file_type = meta.get("file_type", "unknown")
        chunk_id = meta.get("chunk_id", "")
        formatted.append(
            f"[출처: {source} ({file_type}, {chunk_id})]\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(formatted)

5. main.py를 실행해서 키키테크에 대한 정보에 대해 잘 대답하는지 확인해 봅시다.